In [9]:
from pprint import pprint

query = "What alloys were selected for validation and why?"
query_analysis = pipeline.query_analyzer.analyze(query)

retrieval_hits = pipeline.retriever.search(
    query=query_analysis.normalized_query,
    buckets=query_analysis.retrieval_buckets,
    graph=graph,
    top_k=8,
)

expanded_evidence = pipeline.expander.expand(
    hits=retrieval_hits,
    graph=graph,
    query_analysis=query_analysis,
)

print("===== retrieval hits =====")
for h in retrieval_hits[:3]:
    print(h.node_id, h.section_number, h.section_title)

print("\n===== expanded_evidence keys =====")
print(expanded_evidence.keys())

print("\n===== chunk count =====")
print(len(expanded_evidence["chunks"]))

print("\n===== first chunk =====")
pprint(expanded_evidence["chunks"][3], width=120)

print("\n===== all chunk role summary =====")
for i, c in enumerate(expanded_evidence["chunks"][:20], 1):
    print(
        i,
        "node_id =", c.get("node_id"),
        "| role =", c.get("expand_role"),
        "| expanded_from =", c.get("expanded_from"),
        "| section =", c.get("section_number"), c.get("section_title"),
    )

===== retrieval hits =====
sec_4_1_1__chunk_1 4.1.1 Phase equilibrium
sec_4_1_4__chunk_3 4.1.4 Effect on γ′ morphology
sec_4_2__chunk_3 4.2 Features of the designed alloys

===== expanded_evidence keys =====
dict_keys(['chunks', 'figures', 'tables', 'equations', 'references'])

===== chunk count =====
14

===== first chunk =====
{'content': '[EQUATION:eq_1] The γ′ morphology tends to be spherical at near-zero lattice misfit and becomes cuboidal '
            'with higher absolute values of lattice misfit[75]. The γ / γ′ lattice misfits of similar CoNi-based '
            'superalloy compositions are all positive[32]. Therefore, it can be reasonably inferred that the γ / γ′ '
            'lattice misfits of the alloys studied in this work should also be positive values, i. e. the γ′ lattice '
            'parameter is larger than that of γ phase. From the elemental partitioning results in the γ / γ′ '
            'two-phase, as shown in Figs. 12 and 13, Mo and Cr partition to the γ phas

In [10]:
print("\n===== expanded counts =====")
for k, v in expanded_evidence.items():
    print(k, len(v))

print("\n===== references =====")
for x in expanded_evidence["references"][:10]:
    print(x.get("node_id"), x.get("title"), x.get("triggered_by_chunk_ids"))

print("\n===== figures =====")
for x in expanded_evidence["figures"][:10]:
    print(x.get("node_id"), x.get("title"), x.get("triggered_by_chunk_ids"))

print("\n===== tables =====")
for x in expanded_evidence["tables"][:10]:
    print(x.get("node_id"), x.get("title"), x.get("triggered_by_chunk_ids"))


===== expanded counts =====
chunks 14
figures 1
tables 1
equations 0
references 17

===== references =====
ref_11 Reference 11 ['sec_4_1_1__chunk_1']
ref_22 Reference 22 ['sec_4_1_1__chunk_1']
ref_3 Reference 3 ['sec_4_1_1__chunk_1', 'sec_4_1_4__chunk_3']
ref_31 Reference 31 ['sec_4_1_1__chunk_1']
ref_5 Reference 5 ['sec_4_1_1__chunk_1']
ref_55 Reference 55 ['sec_4_1_1__chunk_1']
ref_76 Reference 76 ['sec_4_1_4__chunk_3']
ref_77 Reference 77 ['sec_4_1_4__chunk_3', 'sec_4_1_4__chunk_4']
ref_78 Reference 78 ['sec_4_1_4__chunk_3', 'sec_4_1_4__chunk_4']
ref_30 Reference 30 ['sec_4_2__chunk_3']

===== figures =====
fig_1 Fig. 1 ['sec_2_1__chunk_5']

===== tables =====
table_5 Table 5 ['sec_4_2__chunk_3']


In [1]:
import json
from main_pipeline import PipelineConfig, build_pipeline_from_output_dir

config = PipelineConfig(
    output_dir="output_503",
    embedding_model_path=r"D:\BGE_large_en_1.5v",
    env_file_path=r"C:\Users\12279\ZHIPU.env",
    rerank_model_path=r"D:\BGE-rerank-large",
    device="cuda",
    retrieval_top_k=12,
    rerank_top_n=6,
)

pipeline, graph = build_pipeline_from_output_dir(config)

result = pipeline.run(
    query="How were the diffusion-multiple alloys designed and heat treated in method?",
    graph=graph,
    top_k=8,
)

#print(json.dumps(result["retrieval_debug"], ensure_ascii=False, indent=2))
print(result["answer"])
print(json.dumps(result["claims"], ensure_ascii=False, indent=2))
print(json.dumps(result["render_payload"], ensure_ascii=False, indent=2))
print(result["retrieval_debug"]["llm_context"][:1500])

C:\Users\12279\A一键材料知识宇宙\main_pipeline.py:279: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceBgeEmbeddings(


The diffusion-multiple alloys were designed using CALPHAD and PANDAT™ software to maximize the γ/γ′ two-phase region. They were prepared by arc melting high purity metals under Ar atmosphere, followed by HIP at 1200 °C / 160 MPa for 5 h. The heat treatment involved a two-stage process: 1240 °C / 50 h with air cooling and 850 °C / 1000 h with water cooling.
[
  {
    "claim_id": "claim_1",
    "text": "The diffusion-multiple alloys were designed using CALPHAD and PANDAT™ software.",
    "supports": [
      {
        "evidence_uid": "503::sec_2_1__chunk_1",
        "source": {
          "doc_id": "503",
          "source_file": "503.json",
          "corpus_id": null
        },
        "node_id": "sec_2_1__chunk_1",
        "node_type": "section_chunk",
        "display_label": "sec_2_1__chunk_1",
        "support_role": "primary_text"
      }
    ]
  },
  {
    "claim_id": "claim_2",
    "text": "The alloys were prepared by arc melting high purity metals under Ar atmosphere.",
    "supp

In [3]:
print(result["query_analysis"])

{'original_query': 'How were the diffusion-multiple alloys designed and heat treated in method?', 'normalized_query': 'How were the diffusion-multiple alloys designed and heat treated in the method section?', 'query_type': 'method', 'needs_decomposition': False, 'sub_questions': [], 'retrieval_buckets': ['method'], 'need_figures': False, 'need_tables': False, 'need_equations': False, 'need_references': False, 'vision_required': False, 'vision_reason': ''}


In [5]:
print(result["retrieval_debug"]["llm_context"])

[CHUNK]
node_id: sec_2_1__chunk_6
role: primary_hit
expanded_from: sec_2_1__chunk_6
section: 2.1 Design and preparation of the multicomponent diffusion-multiple
text: Hot isostatic pressing (HIP) at 1200 °C / 160 MPa for 5 h was used to compact the diffusion interfaces and achieve tight interfacial contacts. Subsequently, the diffusion-multiple along with some Ta foil were encapsulated in a quartz tube, which was back-filled with Ar gas. Finally, a two-stage heat treatment process of 1240 °C / 50 h with air cooling and 850 °C / 1000 h with water cooling was performed. As introduced by Cao et al.[35,36], 1240°C/50h was mainly used to form a composition gradient along the couple interfaces quickly at high temperature and 850 °C / 1000 h was used to achieve phase equilibrium at a target service temperature.

[CHUNK]
node_id: sec_2_1__chunk_5
role: context_prev
expanded_from: sec_2_1__chunk_6
section: 2.1 Design and preparation of the multicomponent diffusion-multiple
text: The size of the